<a href="https://colab.research.google.com/github/B1ackA1paca/cu-stip-ai-red/blob/main/stip_red.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets trl peft bitsandbytes accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.8 MB/s eta 0:00:00


In [6]:
import os
import random
import pickle
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import torch.nn.functional as F
import random

In [7]:
device = ("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

## data prepare

In [8]:
train = pd.read_json('kid_adult.jsonl', lines=True)
test = pd.read_json("public_test_style.jsonl", lines=True)

In [9]:
display(train.sample(1), test.sample(1))

,prompt,kid,adult
1373,Что едят божьи коровки? Большинство божьих кор...,Эти маленькие жучки — настоящие помощники в ог...,Рацион большинства божьих коровок состоит из м...


,prompt,kid,adult,fact
27,"Что такое световой год? Это расстояние, которо...","Световой год — это не время, а огромная линейк...",Световой год — это внесистемная единица измере...,"Световой год — это расстояние, которое свет пр..."


In [14]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [16]:
class TrainDataset():
  def __init__(self, data):
    self.data = data
  def __len__(self):
    return len(self.data)
  def __getitem__(self, i):
    return self.data.loc[i, "prompt"], self.data.loc[i, "kid"]
class TestDataset():
  def __init__(self, data):
    self.data = data
  def __len__(self):
    return len(self.data)
  def __getitem__(self, i):
    return self.data.loc[i, "prompt"]

In [17]:
def collate_fn_train(batch):
    texts = []
    for i in batch:
      texts.append(f"<|im_start|>user\n{i[0]}<|im_end|>\n<|im_start|>assistant\n{i[1]}<|im_end|>")

    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

    inputs["labels"] = inputs["input_ids"].clone()
    inputs["labels"][inputs["labels"] == tokenizer.pad_token_id] = -100

    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": inputs["labels"]
    }
def collate_fn_test(batch):
    tokenizer.padding_side = "left"

    texts = []
    for i in batch:
        texts.append(f"<|im_start|>user\n{i}<|im_end|>\n<|im_start|>assistant\n")

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"]
    }

In [18]:
train_part, val_part = train_test_split(train, test_size=0.2)
train_part, val_part = train_part.reset_index(drop=True), val_part.reset_index(drop=True)
train_loader = DataLoader(TrainDataset(train_part), batch_size=16, shuffle=True, collate_fn=collate_fn_train)
test_loader = DataLoader(TrainDataset(val_part), batch_size=16, shuffle=True, collate_fn=collate_fn_train)

## model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    load_in_4bit=True,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)-